In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("geo_data.csv")
data.head()

,Unnamed: 0,address,name_ru,avg_rating,rubrics,lat,lon,geocode_status
0,0,"г. Москва, пр-кт Волгоградский, д. 46 Б",Стратегия Оценки,5.0,Оценочная компания;Экспертиза;Строительная экс...,NaN,NaN,not_found
1,1,",Москва, Западный административный округ, райо...",One and Double,1.0,Кофейня,NaN,NaN,not_found
2,2,"Адрес: Московская область, Люберецкий район, Б...",Газмагистраль,5.0,"АГНС, АГЗС, АГНКС",NaN,NaN,not_found
3,3,Москва,Экспериментальный квартал,5.0,Достопримечательность,55.625578,37.606392,success
4,4,Москва,ОкнаДел,5.0,Окна;Изготовление витражей;Реставрационная мас...,55.625578,37.606392,success


In [3]:
data = data.dropna(subset=["lat", "lon", "rubrics"])
data["rub_list"] = (
    data["rubrics"]
        .astype(str)
        .str.split(";")
        .apply(lambda x: [s.strip() for s in x if s.strip()])
)
data = data[data["address"] != "Москва"]
data.head()

,Unnamed: 0,address,name_ru,avg_rating,rubrics,lat,lon,geocode_status
16,16,"Москва, 1-й Автозаводский проезд, 4к1",Абрикосик,2.0,Массажный салон,55.703938,37.656436,success
17,17,"Москва, 1-й Автозаводский проезд, 5",Чайхана Азия,2.0,Кафе,55.704831,37.657289,success
18,18,"Москва, 1-й Автозаводский проезд, 5",RenarDance,5.0,Школа танцев,55.704831,37.657289,success
20,20,"Москва, 1-й Амбулаторный проезд, 8с1",Колледж автомобильного транспорта № 9,5.0,Колледж,55.811594,37.532844,success
21,21,"Москва, 1-й Балтийский переулок, 3/25",ХуанХэ,5.0,Ресторан;Кафе,55.810501,37.518897,success


In [4]:
data = data.drop("geocode_status", axis = 1)

Index(['Unnamed: 0', 'address', 'name_ru', 'avg_rating', 'rubrics', 'lat',
       'lon'],
      dtype='object')

In [ ]:
from collections import Counter

all_rubrics = Counter(x for lst in data["rub_list"] for x in lst)
keep_rubrics = {k for k, v in all_rubrics.items() if v >= 50}

In [ ]:
import numpy as np
from sklearn.neighbors import BallTree

EARTH_R = 6371000.0

coords_rad = np.radians(data[["lat", "lon"]].values)
tree = BallTree(coords_rad, metric="haversine")

In [ ]:
def count_rubrics_for_point(lat, lon, R_m):
    idxs = tree.query_radius(
        np.radians([[lat, lon]]),
        r=R_m / EARTH_R
    )[0]

    cnt = {}
    for i in idxs:
        for r in data.iloc[i]["rub_list"]:
            if r in keep_rubrics:
                cnt[r] = cnt.get(r, 0) + 1

    return cnt

In [ ]:
R = 1000  # радиус в метрах

features = []
for _, row in df.iterrows():
    features.append(
        count_rubrics_for_point(row["lat"], row["lng"], R)
    )

rub_df = pd.DataFrame(features).fillna(0).astype(int)
rub_df.columns = [f"cnt_{c.replace(' ', '_')}" for c in rub_df.columns]

df = pd.concat([df.reset_index(drop=True),
                rub_df.reset_index(drop=True)], axis=1)


In [5]:
from geopy.distance import EARTH_RADIUS


def haversin(theta):
    return (1 - np.cos(theta)) / 2.0

def haversine_2D_mat(data1, data2):
    lats1, lons1 = data1['lat'].values, data1['lon'].values
    phis1, lambs1 = np.radians(lats1).reshape(-1, 1), np.radians(lons1).reshape(-1, 1)

    lats2, lons2 = data2['lat'].values, data2['lon'].values
    phis2, lambs2 = np.radians(lats2).reshape(-1, 1), np.radians(lons2).reshape(-1, 1)

    deltas_lats = phis1 - phis2.T
    deltas_lons = lambs1 - lambs2.T

    cos_phis1 = np.cos(phis1)
    cos_phis2 = np.cos(phis2)
    a = haversin(deltas_lats) + cos_phis1 * cos_phis2.T * haversin(deltas_lons)

    vec_dist = 2 * EARTH_RADIUS * np.arcsin(np.sqrt(a))
    return vec_dist * 1000

In [6]:
def get_list(lat, lon, R, rub_groups):
    point_df = pd.DataFrame({"lat": [lat], "lon": [lon]})
    ls = haversine_2D_mat(point_df, df)[0]

    r = df.loc[ls <= R, "rubrics"].dropna()
    rub_split = r.str.split(";").apply(set)

    res = {}
    for group in rub_groups:
        key = "_".join(group)
        cnt = 0
        group_set = set(group)
        for s in rub_split:
            if s & group_set:
                cnt += 1
        res[key] = cnt
    return res


In [7]:
R = 1000
rub_lst = get_list(55.810501, 37.518897, R, [["Ресторан", "Кафе"], ["Кафе"], ["Ресторан"]])
rub_lst

{'Ресторан_Кафе': 10, 'Кафе': 8, 'Ресторан': 6}